# Lab | Pandas

## Insurance customer data analysis

In this lab we load customer data from an insurance company and use pandas to explore, clean, and analyze it.

### Data description

- Customer: customer ID
- ST: state where customers live
- GENDER: customer gender
- Education: education level
- Customer Lifetime Value: estimated customer lifetime value
- Income: customer income
- Monthly Premium Auto: monthly insurance premium
- Number of Open Complaints: number of complaints opened
- Policy Type: Corporate Auto, Personal Auto, or Special Auto
- Vehicle Class: customer vehicle class
- Total Claim Amount: total value of claims made by the customer

## Challenge 1: Understanding the data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv"
raw_df = pd.read_csv(url)

# The source file contains blank rows, so remove rows that are empty in every column.
df = raw_df.dropna(how="all").copy()

# Correct two columns that are encoded as text in the CSV.
df["Customer Lifetime Value"] = (
    df["Customer Lifetime Value"].str.replace("%", "", regex=False).astype(float)
)
df["Number of Open Complaints"] = (
    df["Number of Open Complaints"]
    .str.extract(r"^1/(\d+)/", expand=False)
    .astype("Int64")
)

print(f"Raw dimensions: {raw_df.shape}")
print(f"Clean dimensions: {df.shape}")
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())

The raw file contains blank rows that should be removed. Customer Lifetime Value should be numeric after removing the percent sign. Number of Open Complaints should be an integer; the values in the source are formatted like dates, so the number between the slashes is extracted. ST, GENDER, Education, Policy Type, and Vehicle Class are categorical columns. Customer is an identifier and is best treated as a categorical/string field.

In [ ]:
categorical_columns = [
    "Customer", "ST", "GENDER", "Education", "Policy Type", "Vehicle Class"
]
numeric_columns = [
    "Customer Lifetime Value", "Income", "Monthly Premium Auto",
    "Number of Open Complaints", "Total Claim Amount"
]

print("Unique values per column:")
print(df.nunique(dropna=False))

print("\nCategorical values:")
for column in categorical_columns:
    print(f"\n{column}:")
    print(df[column].value_counts(dropna=False))

print("\nNumerical ranges:")
for column in numeric_columns:
    print(f"{column}: {df[column].min()} to {df[column].max()}")

In [ ]:
print("Summary statistics for numerical columns:")
numeric_summary = df[numeric_columns].describe().T
numeric_summary["median"] = df[numeric_columns].median()
numeric_summary["mode"] = df[numeric_columns].mode().iloc[0]
print(numeric_summary)

print("\nSummary statistics for categorical columns:")
print(df[categorical_columns].describe().T)

The numerical summaries show that Income and Customer Lifetime Value vary substantially between customers. Monthly Premium Auto and Total Claim Amount are right-skewed because their means are above their medians and their maximum values are much higher than their upper quartiles. The categorical summaries show that Personal Auto is the most common policy type, and that the state and gender fields contain inconsistent labels such as Cali/California, WA/Washington, AZ/Arizona, Male/M, and several female spellings. These labels could be standardized before a more detailed analysis.

## Challenge 2: Analyzing the data

### Exercise 1

The marketing team wants to know the five least common customer locations.

In [ ]:
customer_locations = df["ST"].value_counts()
least_common_locations = customer_locations.sort_values().head(5)

print("Customer locations and frequencies:")
print(customer_locations)
print("\nFive least common locations:")
print(least_common_locations)

### Exercise 2

The sales team wants the number of policies sold for each policy type and the most common policy type.

In [ ]:
policy_type_counts = df["Policy Type"].value_counts()
most_common_policy_type = policy_type_counts.idxmax()

print("Policies sold by policy type:")
print(policy_type_counts)
print(f"\nPolicy type with the most policies sold: {most_common_policy_type}")

### Exercise 3

Compare the average income of customers with Personal Auto and Corporate Auto policies.

In [ ]:
personal_auto = df.loc[df["Policy Type"] == "Personal Auto"]
corporate_auto = df.loc[df["Policy Type"] == "Corporate Auto"]

personal_average_income = personal_auto["Income"].mean()
corporate_average_income = corporate_auto["Income"].mean()

print(f"Average income - Personal Auto: {personal_average_income:,.2f}")
print(f"Average income - Corporate Auto: {corporate_average_income:,.2f}")

if personal_average_income < corporate_average_income:
    print("Personal Auto customers have a lower average income.")
else:
    print("Personal Auto customers do not have a lower average income.")

### Bonus: Exercise 4

Identify customers whose Total Claim Amount is in the top 25% of the distribution.

In [ ]:
claim_amount_75th_percentile = df["Total Claim Amount"].quantile(0.75)
high_claim_customers = df.loc[
    df["Total Claim Amount"] > claim_amount_75th_percentile
].copy()

print(f"75th percentile of Total Claim Amount: {claim_amount_75th_percentile:.2f}")
print(f"Number of high-claim customers: {len(high_claim_customers)}")
print("\nHigh-claim customer data:")
print(high_claim_customers.head())
print("\nSummary statistics for high-claim customers:")
print(high_claim_customers.describe(include="all").T)